


###Desenvolvimento de projeto ETL em Python para integração e consolidação de dados provenientes de diferentes APIs e plataformas



### Bibliotecas

In [0]:
import requests
import pandas as pd

### Api github + Endpoints

In [0]:
# especificando a versão da API
headers = {'X-GitHub-Api-Version': '2022-11-28'}

In [0]:
r = requests.get('https://api.github.com/events', headers=headers)

In [0]:
r= requests.get('https://api.github.com/events')
r.status_code

In [0]:
r.json()

In [0]:
r= requests.get('https://api.github.com/versions')
r.status_code

### Obtendo dados dos repositórios

In [0]:
# especificando a versão da API
headers = {'X-GitHub-Api-Version': '2022-11-28'}

In [0]:
### chamando os dados do repositorio da amazon 
api_base_url = 'https://api.github.com'
owner = 'amzn' ## User que eu vou buscar no repositorio
url = f'{api_base_url}/users/{owner}/repos'

In [0]:
###verificar o status da minha api
response = requests.get(url, headers=headers)
response.status_code

In [0]:
response.json()

In [0]:
###Autentificação da API
acess_token ='xx'
headers = {'Authorization': 'Bearer' + acess_token,
           'X-GitHub-Api-Version': '2022-11-28'}


In [0]:
api_base_url = 'https://api.github.com'
owner = 'amzn' ## User que eu vou buscar no repositorio
url = f'{api_base_url}/users/{owner}/repos'

url

In [0]:
##paginação solitar dentro do api

repos_list = []

for page_num in range(1, 6):
    try:
        url_page = f'{url}?page={page_num}'
        response = requests.get(url_page, headers=headers)
        repos_list.append(response.json())
    except:
        repos_list.append(None)


In [0]:
repos_list 

In [0]:
###quantidade de paginas 
len(repos_list)

### Transformando dados 

In [0]:
##identificar quais dados estão no repositorio
repos_list[0][2]['name'] #< pagina e repositorio

In [0]:
repos_list

In [0]:
###escolhendo pagina e repositorio
repos_list [0][2]['name']

In [0]:
###Colocando em uma lista e nome dos respositorios que existe no github

respos_name= []
for page in repos_list:
   for repo in page :
    respos_name.append(repo ['name'])

In [0]:
##visualiza os 10 primeiros
respos_name[:10]

In [0]:
###Colocando em uma lista e nome das linguaguem de programacao que existe no github

respos_language= []
for page in repos_list:
   for repo in page :
    respos_language.append(repo ['language'])

In [0]:
##extracao das 10primeiras linguagens
respos_language[:10]

### Criando um dataframe

In [0]:
###Criar dataframe 
dados_amz = pd.DataFrame ()
dados_amz['nome']= respos_name
dados_amz['linguagem']= respos_language

In [0]:
dados_amz

### Salvando CSV

In [0]:
dados_amz.to_csv('amazon.csv')

### Repostorio POST

In [0]:
###chamando dados da API

api_base_url = "https://api.github.com"
url = f'{api_base_url}/user/repos'

url

In [0]:
####criando um dicionario para dados 

data={
   'name':'linguagens-utilizadas',
    'description':'Repositorio com as linguaguens de programacao amazon',
    'private': False

}

response= requests.post(url,json= data, headers=headers)
response.status_code

### formato do arquivo

In [0]:
import base64

with open('amazon.csv', 'rb') as file:
    file_content = file.read()

encoded_content = base64.b64encode(file_content).decode('utf-8')

print(encoded_content)

In [0]:

##Uploaud de arquivo com put
api_base_url = 'https://api.github.com'
username = 'millenagena'
repo = 'linguagens-utilizadas'
path = 'amazon.csv'

url = f'{api_base_url}/repos/{username}/{repo}/contents/{path}'
url

###Dados Repos

In [0]:
import requests
import pandas as pd

class DadosRepositorios:

    def __init__(self, owner):
        self.owner = owner
        self.api_base_url = 'https://api.github.com'
        self.access_token='seu_token' 
        self.headers = {'Authorization':'Bearer ' + self.access_token,
                        'X-GitHub-Api-Version': '2022-11-28'}

    def lista_repositorios(self):
        repos_list = []

        for page_num in range(1, 20):
            try:
                url = f'{self.api_base_url}/users/{self.owner}/repos?page={page_num}'
                response = requests.get(url, headers=self.headers)
                repos_list.append(response.json())
            except:
                repos_list.append(None)
        
        return repos_list
    
    def nomes_repos(self, repos_list):
        repo_names=[]
        for page in repos_list:
            for repo in page:
                try:
                    repo_names.append(repo['name'])
                except:
                    pass

        return repo_names
    
    def nomes_linguagens(self, repos_list):
        repo_languages=[]
        for page in repos_list:
            for repo in page:
                try:
                    repo_languages.append(repo['language'])
                except:
                    pass

        return repo_languages
    
    def cria_df_linguagens(self):

        repositorios = self.lista_repositorios()
        nomes = self.nomes_repos(repositorios)
        linguagens = self.nomes_linguagens(repositorios)

        dados = pd.DataFrame()
        dados['repository_name'] = nomes
        dados['language'] = linguagens

        return dados  
    
amazon_rep = DadosRepositorios('amzn')
ling_mais_usadas_amzn = amazon_rep.cria_df_linguagens()
#print(ling_mais_usadas_amzn)

netflix_rep = DadosRepositorios('netflix')
ling_mais_usadas_netflix = netflix_rep.cria_df_linguagens()

spotify_rep = DadosRepositorios('spotify')
ling_mais_usadas_spotify = spotify_rep.cria_df_linguagens()

# Salvando os dados 
ling_mais_usadas_amzn.to_csv('dados/linguagens_amzn.csv')
ling_mais_usadas_netflix.to_csv('dados/linguagens_netflix.csv')
ling_mais_usadas_spotify.to_csv('dados/linguagens_spotify.csv')